# Notebook 01 — Signal Analysis
**DSP501 — Speaker Identification**

This notebook visualizes:
- Raw waveform vs filtered waveform
- FFT magnitude spectrum
- STFT spectrogram
- Power Spectral Density (PSD)
- SNR before/after filtering
- FIR filter frequency & phase response

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt

from preprocess import preprocess
from filter import design_fir, apply_filter, plot_frequency_response, plot_phase_response
from analysis import plot_waveform, plot_spectrum, plot_stft, compute_psd, compute_snr

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load one sample audio file

In [ ]:
# Change this path to any .wav file in your dataset
SAMPLE = '../data/raw/speaker_01/01.wav'
SR = 16000

y_raw, sr = preprocess(SAMPLE, sr=SR)
print(f'Signal length : {len(y_raw)} samples = {len(y_raw)/SR:.2f} s')
print(f'Sample rate   : {sr} Hz')

## 2. Design FIR Bandpass Filter (300–3400 Hz)

In [ ]:
coeffs = design_fir(lowcut=300, highcut=3400, sr=SR, numtaps=101)
print(f'Filter taps: {len(coeffs)}')
print(f'First 5 coefficients: {coeffs[:5].round(6)}')

In [ ]:
# Frequency response
plot_frequency_response(coeffs, sr=SR, save_path='../figures/freq_response.png')

In [ ]:
# Phase response (should be linear — key advantage of FIR)
plot_phase_response(coeffs, sr=SR, save_path='../figures/phase_response.png')

## 3. Apply Filter

In [ ]:
y_filt = apply_filter(y_raw, coeffs)
print('Filtering done.')

## 4. Waveform Comparison

In [ ]:
plot_waveform(y_raw, y_filt, sr=SR, save_path='../figures/waveform_comparison.png')

## 5. FFT Spectrum Comparison

In [ ]:
plot_spectrum(y_raw, y_filt, sr=SR, save_path='../figures/spectrum_comparison.png')

## 6. STFT Spectrogram Comparison

In [ ]:
plot_stft(y_raw, y_filt, sr=SR, save_path='../figures/stft_comparison.png')

## 7. Power Spectral Density (PSD)

In [ ]:
freqs_r, psd_r = compute_psd(y_raw, SR)
freqs_f, psd_f = compute_psd(y_filt, SR)

plt.figure(figsize=(9, 4))
plt.semilogy(freqs_r, psd_r, label='Raw')
plt.semilogy(freqs_f, psd_f, label='Filtered', color='orange')
plt.axvline(300,  color='r', linestyle='--', linewidth=0.8)
plt.axvline(3400, color='g', linestyle='--', linewidth=0.8)
plt.title('Power Spectral Density')
plt.xlabel('Frequency (Hz)')
plt.ylabel('PSD')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('../figures/psd_comparison.png', dpi=150)
plt.show()

## 8. SNR Before / After Filtering

In [ ]:
snr = compute_snr(y_raw, y_filt)
print(f'SNR improvement after FIR filtering: {snr:.2f} dB')
print('(Higher = more noise removed relative to kept signal)')